# 13 baby pose 학습 — Colab (GPU · 대량 epoch)

로컬 `3_run_training.py` 와 **동일 설정**(회전/flip 증강, **Pose 우선 fitness**, 에폭별 best 표시, best 리포트)을
Colab GPU에서 EPOCHS 크게 돌리는 노트북. **self-contained** (folder 3 코드 불필요).

**순서**
1. 상단 메뉴 **런타임 > 런타임 유형 변경 > 하드웨어 가속기: GPU**.
2. 아래 셀을 위→아래 순서로 실행.

**데이터 준비 (로컬 PowerShell)** — dataset 폴더를 zip 으로:

    Compress-Archive -Path D:\Code\Monitoring\IoT-Monitoring\media\13_pose_train\dataset -DestinationPath D:\dataset.zip

→ 생성된 `dataset.zip` 을 아래 업로드 셀에서 선택. (라벨을 더 만들면 다시 zip → 재업로드 → 재학습)

In [ ]:
!pip -q install ultralytics
import ultralytics; ultralytics.checks()

## 1) 데이터 업로드
`dataset.zip` 선택 → `/content` 에 풀고 images/labels 폴더 자동 탐색. (대안: Google Drive 마운트 — 셀 하단 주석)

In [ ]:
import os, glob, shutil, zipfile
from google.colab import files

up = files.upload()                          # dataset.zip 선택
zname = next(iter(up))
shutil.rmtree('/content/ds', ignore_errors=True)
with zipfile.ZipFile(zname) as z:
    z.extractall('/content/ds')

def _find(root):                             # images/labels 가 함께 있는 폴더 탐색
    for d, _, _ in os.walk(root):
        if os.path.isdir(os.path.join(d, 'images')) and os.path.isdir(os.path.join(d, 'labels')):
            return d
    return None

DATA_ROOT = _find('/content/ds')
assert DATA_ROOT, 'images/labels 폴더를 못 찾음 — zip 구조 확인'
print('dataset:', DATA_ROOT,
      '| images', len(glob.glob(DATA_ROOT + '/images/*.jpg')),
      '| labels', len(glob.glob(DATA_ROOT + '/labels/*.txt')))

# --- 대안: Google Drive 사용 시 위 업로드 대신 ---
# from google.colab import drive; drive.mount('/content/drive')
# DATA_ROOT = '/content/drive/MyDrive/pose_dataset'   # images/labels 있는 폴더

## 2) data.yaml + train/val split (clip 단위, 누수 방지)
로컬 `2_build_data.py` 와 동일 로직 인라인. 파일은 그대로 두고 리스트/yaml 만 생성.

In [ ]:
import random
VAL_RATIO, SEED = 0.2, 0
FLIP_IDX = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]   # COCO 좌우반전
NUM_KPTS = 17

imgs = [p for p in sorted(glob.glob(os.path.join(DATA_ROOT, 'images', '*.jpg')))
        if os.path.exists(os.path.join(DATA_ROOT, 'labels',
                                       os.path.splitext(os.path.basename(p))[0] + '.txt'))]
clips = {}
for p in imgs:                                # clip = '{영상}_f0000' 의 _f 앞부분
    clips.setdefault(os.path.splitext(os.path.basename(p))[0].rsplit('_f', 1)[0], []).append(p)
keys = sorted(clips); random.Random(SEED).shuffle(keys)
n_val = round(len(keys) * VAL_RATIO) if len(keys) > 1 else 0
val_keys = set(keys[:n_val])
train = [p for k in keys if k not in val_keys for p in clips[k]]
val   = [p for k in val_keys for p in clips[k]]

with open(os.path.join(DATA_ROOT, 'train.txt'), 'w') as f:
    f.write("\n".join(train) + "\n")
with open(os.path.join(DATA_ROOT, 'val.txt'), 'w') as f:
    f.write(("\n".join(val) + "\n") if val else "")
flip = ", ".join(map(str, FLIP_IDX))
with open(os.path.join(DATA_ROOT, 'data.yaml'), 'w') as f:
    f.write(f"path: {DATA_ROOT}\ntrain: train.txt\nval: {'val.txt' if val else 'train.txt'}\n\n"
            f"kpt_shape: [{NUM_KPTS}, 3]\nflip_idx: [{flip}]\n\nnames:\n  0: baby\n")
DATA_YAML = os.path.join(DATA_ROOT, 'data.yaml')
print(f'clips {len(keys)} · imgs {len(imgs)} -> train {len(train)} / val {len(val)}')
print(open(DATA_YAML).read())

## 3) 학습 (Pose 우선 fitness · 에폭별 best 표시)
`EPOCHS` 크게. OOM 이면 `BATCH` 를 8 로. 조기종료 끄려면 `PATIENCE=0`.

In [ ]:
# ===== 설정 (로컬 3_run_training.py 와 동일 개념) =====
BASE_MODEL = 'yolo11n-pose.pt'
EPOCHS   = 1000
PATIENCE = 50            # Pose mAP 노이즈 고려 여유있게 (0=조기종료 끄기)
IMGSZ    = 640
BATCH    = 16            # Colab GPU 여유 → 키움 (OOM 이면 8)
DEGREES  = 45.0
FLIPLR, FLIPUD = 0.5, 0.5
POSE_FITNESS_W = 0.9     # fitness = W*Pose + (1-W)*Box  (best.pt·조기종료 Pose 우선, 1.0=순수 Pose)
AMP = True               # Colab GPU 는 fp16 안정 → 속도 up (로컬 GTX1650 만 False 였음)
OUTPUT, RUN = '/content/runs', 'train'

from ultralytics import YOLO
from ultralytics.utils.metrics import DetMetrics, PoseMetrics

def _fit(self):                                              # Pose 우선 fitness
    return POSE_FITNESS_W * self.pose.fitness() + (1 - POSE_FITNESS_W) * DetMetrics.fitness.fget(self)
PoseMetrics.fitness = property(_fit)

def _marker(tr):                                             # 에폭별 best + Pose mAP
    if tr.fitness is None:
        return
    m = getattr(tr, 'metrics', None) or {}
    pose = m.get('metrics/mAP50-95(P)')
    ex = f'  Pose50-95={float(pose):.4f}' if pose is not None else ''
    best = float(tr.best_fitness) if tr.best_fitness is not None else float(tr.fitness)
    mk = '  ★ NEW BEST' if float(tr.fitness) >= best else f'  (best {best:.4f})'
    print(f'[epoch {tr.epoch + 1}] fitness={float(tr.fitness):.4f}{ex}{mk}', flush=True)

model = YOLO(BASE_MODEL)
model.add_callback('on_fit_epoch_end', _marker)
model.train(data=DATA_YAML, epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
            device=0, amp=AMP, degrees=DEGREES, fliplr=FLIPLR, flipud=FLIPUD,
            project=OUTPUT, name=RUN, exist_ok=True)

## 4) best 에폭 리포트 + best.pt 내려받기

In [ ]:
import csv
TRAIN_DIR = os.path.join(OUTPUT, RUN)

def report_best(train_dir):
    rows = [{k.strip(): v for k, v in r.items()}
            for r in csv.DictReader(open(os.path.join(train_dir, 'results.csv')))]
    def fitf(r):
        return (POSE_FITNESS_W * float(r['metrics/mAP50-95(P)'])
                + (1 - POSE_FITNESS_W) * float(r['metrics/mAP50-95(B)']))
    i = max(range(len(rows)), key=lambda j: fitf(rows[j])); r = rows[i]
    print(f"best epoch {int(float(r['epoch']))}/{len(rows)}   "
          f"Pose mAP50-95={float(r['metrics/mAP50-95(P)']):.4f} mAP50={float(r['metrics/mAP50(P)']):.4f}   "
          f"Box mAP50-95={float(r['metrics/mAP50-95(B)']):.4f}")

report_best(TRAIN_DIR)
from google.colab import files
files.download(os.path.join(TRAIN_DIR, 'weights', 'best.pt'))    # 로컬로 다운로드
# 결과 그래프도 필요하면:  files.download(os.path.join(TRAIN_DIR, 'results.png'))

## 5) 내려받은 best.pt 로컬에서 쓰기
`best.pt` 를 `13_pose_train\output\colab\weights\best.pt` 로 넣고:

    cd D:\Code\Monitoring\IoT-Monitoring\media\13_pose_train
    ..\.venv\Scripts\python.exe 4_Pose_Sample_Test.py --model output\colab\weights\best.pt --src D:\carved-08\Cut

또는 `1_pose_correct.py --model output\colab\weights\best.pt` 로 이 모델 예측을 프리필해 라벨을 더 교정.